# 00.1 · Solutions

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/cloudops-one/training-ai-ml/blob/main/modules/00-big-picture/01-why-machine-learning/solutions.ipynb)

In [1]:
import pandas as pd

orders = pd.DataFrame({
    "total":          [45, 620, 310, 880, 95, 540, 430, 70],
    "account_days":   [400, 3, 120, 1, 2, 800, 5, 30],
    "items":          [2, 9, 4, 12, 1, 6, 7, 3],
    "country_match":  [1, 0, 1, 1, 1, 0, 0, 1],   # 1 = shipping country matches card country
    "fraud":          [0, 1, 0, 1, 0, 0, 1, 0],
})
orders

,total,account_days,items,country_match,fraud
0,45,400,2,1,0
1,620,3,9,0,1
2,310,120,4,1,0
3,880,1,12,1,1
4,95,2,1,1,0
5,540,800,6,0,0
6,430,5,7,0,1
7,70,30,3,1,0


## Exercise 1

In [2]:
extra = pd.DataFrame({
    "total":         [350, 650],   # fraud under the £400 line; honest big order from a new account
    "account_days":  [2, 4],
    "items":         [5, 8],
    "country_match": [0, 1],
    "fraud":         [1, 0],
})
orders2 = pd.concat([orders, extra], ignore_index=True)

def hand_rule(row):
    return int(row["total"] > 400 and row["account_days"] < 7)

orders2["hand_pred"] = orders2.apply(hand_rule, axis=1)
print(f"accuracy: {(orders2['hand_pred'] == orders2['fraud']).mean():.0%}")
orders2.tail(2)

accuracy: 80%


,total,account_days,items,country_match,fraud,hand_pred
8,350,2,5,0,1,0
9,650,4,8,1,0,1


Both new rows are wrong. To fix the first you would lower the £400 threshold, which then flags more honest orders. To fix the second you need another feature (perhaps `country_match`). Every fix moves the problem somewhere else, which is the lesson.

## Exercise 2

In [3]:
import numpy as np
rng = np.random.default_rng(0)

def make_orders(n):
    """Generate n ShopWave orders with the same hidden pattern as the 8 above."""
    total = rng.gamma(2.0, 150, n).round(0)
    account_days = rng.integers(1, 1000, n)
    account_days = np.where(rng.random(n) < 0.25, rng.integers(1, 15, n), account_days)  # a quarter are new accounts
    items = np.clip((total / 70 + rng.normal(0, 1.5, n)).round(), 1, None).astype(int)
    country_match = rng.choice([0, 1], n, p=[0.15, 0.85])
    # hidden truth (the model never sees this line): big order from a new account,
    # OR a very big order shipping to a country that does not match the card.
    # 3% of labels are flipped at random, because real data is never clean.
    truth = ((total > 300) & (account_days < 10)) | ((country_match == 0) & (total > 500))
    fraud = (truth ^ (rng.random(n) < 0.03)).astype(int)
    return pd.DataFrame({"total": total, "account_days": account_days,
                         "items": items, "country_match": country_match, "fraud": fraud})

many = make_orders(400)
print(many.shape)
print("fraud rate:", many["fraud"].mean().round(3))
many.head()

(400, 5)
fraud rate: 0.11


,total,account_days,items,country_match,fraud
0,275.0,281,2,1,0
1,396.0,227,4,1,0
2,160.0,208,2,1,0
3,597.0,33,10,1,0
4,137.0,3,3,1,0


In [4]:
from sklearn.tree import DecisionTreeClassifier, export_text
features = ["total", "account_days", "items", "country_match"]

for depth in [2, 3, 5]:
    t = DecisionTreeClassifier(max_depth=depth, random_state=0).fit(many[features], many["fraud"])
    acc = (t.predict(many[features]) == many["fraud"]).mean()
    print(f"max_depth={depth}: accuracy {acc:.1%}, leaves {t.get_n_leaves()}")

print()
print(export_text(DecisionTreeClassifier(max_depth=5, random_state=0)
                  .fit(many[features], many["fraud"]), feature_names=features))

max_depth=2: accuracy 95.2%, leaves 4
max_depth=3: accuracy 96.5%, leaves 7
max_depth=5: accuracy 97.8%, leaves 17

|--- account_days <= 9.50
|   |--- total <= 297.00
|   |   |--- account_days <= 8.50
|   |   |   |--- items <= 3.50
|   |   |   |   |--- class: 0
|   |   |   |--- items >  3.50
|   |   |   |   |--- total <= 125.00
|   |   |   |   |   |--- class: 1
|   |   |   |   |--- total >  125.00
|   |   |   |   |   |--- class: 0
|   |   |--- account_days >  8.50
|   |   |   |--- total <= 183.50
|   |   |   |   |--- class: 0
|   |   |   |--- total >  183.50
|   |   |   |   |--- class: 1
|   |--- total >  297.00
|   |   |--- class: 1
|--- account_days >  9.50
|   |--- country_match <= 0.50
|   |   |--- total <= 511.00
|   |   |   |--- items <= 2.50
|   |   |   |   |--- total <= 240.00
|   |   |   |   |   |--- class: 0
|   |   |   |   |--- total >  240.00
|   |   |   |   |   |--- class: 1
|   |   |   |--- items >  2.50
|   |   |   |   |--- class: 0
|   |   |--- total >  511.00
|   |   |

Accuracy rises with depth, but look at the deeper rules: many of the extra splits separate only one or two rows, with thresholds like `account_days > 949` that have no business meaning. The tree is memorising the 3% of labels that were flipped at random. Scored on *new* orders it would do worse, not better. That is **overfitting**; Module 04 shows how to detect and prevent it.

## Exercise 3

1. **Rules.** The policy is known and must apply exactly.
2. **ML.** There is no `if` for "looks like a mug"; you need examples.
3. **ML.** Nobody can write the pattern, but past tickets with outcomes exist.
4. **Rules.** Must be guaranteed, legally. A probability is not acceptable.
5. **ML.** Preferences are learned from what similar customers bought (Module 06).